# Fluxo e posições em opções de ações — B3

Protótipo que lê o negócio a negócio e as posições em aberto do Boletim Diário do Mercado e destaca:

- estruturas multi-perna e blocos grandes, lidos do ponto de vista da corretora que não é formador nem alto giro;
- negócios diretos (mesma corretora nos dois lados);
- variação das posições em aberto por componente (coberta, travada, descoberta) e entradas concentradas;
- posicionamento no vencimento mais próximo.

**Dados.** A página do boletim restringe o uso à consulta informativa e veda coleta automatizada. Para prototipar, baixe os arquivos manualmente; para produção e histórico, use dado licenciado, alinhado com compliance.

**Arquivos por pregão**, em `dados/AAAA-MM-DD/` (nomes livres; o tipo é detectado pelo cabeçalho): do capítulo **Renda variável**, cadastro de instrumentos, negócios consolidados do pregão e negócio a negócio; do capítulo **Derivativos de bolsa**, posições em aberto. Cadastro, consolidado e negócio a negócio de Derivativos de bolsa (ex.: `..._NEGOCIOSAVISTA_DRV.txt`) não têm opções de ações. Do pregão anterior basta o arquivo de posições em aberto, que habilita a variação por componente e do número de clientes.

Comece com `MODO_SINTETICO = True` para ver tudo funcionando com dados fictícios.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
def _br(v, nd):   # 1.234.567,89
    return f"{v:,.{nd}f}".replace(",", "X").replace(".", ",").replace("X", ".")

pd.set_option("display.float_format", lambda v: _br(v, 4) if abs(v) < 1 else _br(v, 2) if abs(v) < 1e6 else _br(v, 0))

%load_ext autoreload
%autoreload 2

from fluxo_opcoes import Parametros, processar_dia, exportar_excel, inventariar
import fluxo_opcoes
print(f"fluxo_opcoes {fluxo_opcoes.__version__} — {Path(fluxo_opcoes.__file__).parent}")
from fluxo_opcoes.config import CORRETORAS

MODO_SINTETICO = True          # False para rodar com os arquivos reais
PASTA_DADOS = Path("dados")
DATA = "2026-11-16"            # pregão analisado (pasta PASTA_DADOS/AAAA-MM-DD)
DATA_ANTERIOR = None           # None = usa a pasta de pregão anterior mais recente (recomendado)
OVERRIDES = None               # ex.: {"posicoes": {"n_clientes_1": "Qtd. Titulares"}}
PAINEL_TEXTO_BR = False        # True: números do Excel já gravados no padrão brasileiro (como texto)
CURVA_DI = "pyield"            # "pyield": curva do DI1 do dia; None: usa params.taxa_pre_aa; ou um dict {DU: taxa}

params = Parametros()          # ex.: Parametros(min_delta_notional=10e6, taxa_pre_aa=0.145)


def mostrar(df, cols):
    return df[[c for c in cols if c in df.columns]]


pd.DataFrame(params.como_tabela(), columns=["parâmetro", "valor", "descrição"])

In [ ]:
if MODO_SINTETICO:
    from fluxo_opcoes.sintetico import gerar
    info = gerar("dados_sinteticos", data=DATA, data_anterior=DATA_ANTERIOR or "2026-11-13")
    PASTA_DADOS = Path("dados_sinteticos")
    corretoras = info["corretoras"]      # mapeamento fictício, só para demonstrar
    print("Eventos plantados no pregão sintético:")
    display(pd.DataFrame(info["esperado"], columns=["tipo", "ativo", "corretora", "estrutura"]))
else:
    corretoras = CORRETORAS              # preencha os códigos em fluxo_opcoes/config.py

from fluxo_opcoes.pipeline import pregao_anterior
pasta_dia = PASTA_DADOS / DATA
pasta_ant = PASTA_DADOS / DATA_ANTERIOR if DATA_ANTERIOR else pregao_anterior(PASTA_DADOS, DATA)
print("Pregão analisado:", pasta_dia.name, "| pregão anterior:", pasta_ant.name if pasta_ant else "não encontrado")

## 1. Inspeção dos arquivos

Confira o tipo detectado, a cobertura (1,0 = todos os campos mínimos encontrados) e o mapeamento de colunas. Na primeira rodada com dados reais, verifique em especial qual campo de clientes (`n_clientes_1`/`n_clientes_2`) corresponde a titulares e a lançadores, se tipo e strike vieram do cadastro e se a hora tem milissegundos.

In [ ]:
inv = inventariar(pasta_dia, OVERRIDES)
inv[["arquivo", "tipo", "tipo_provavel", "faltando", "cobertura", "separador", "linha_cabecalho", "mapeadas", "ignoradas"]]

## 2. Processamento do pregão

In [ ]:
from fluxo_opcoes.validacao import carregar_historico, adtv_historico

PASTA_HIST = "historico_sintetico" if MODO_SINTETICO else "historico"
adtv = None
if Path(PASTA_HIST, "precos.csv").exists():      # com 3+ pregões anteriores, ADTV = média (até 20)
    h = carregar_historico(PASTA_HIST)
    anteriores = h["precos"].loc[h["precos"]["data"] < pd.Timestamp(DATA), "data"].nunique()
    if anteriores >= 3:
        adtv = adtv_historico(h["precos"], janela=20, ate=DATA)
        print(f"ADTV = média de {min(anteriores, 20)} pregões anteriores")

curva = None                                     # curva pré: Black-Scholes e spread do financiamento
if isinstance(CURVA_DI, dict):
    curva = CURVA_DI
elif CURVA_DI == "pyield" and not MODO_SINTETICO:
    from fluxo_opcoes.taxas import curva_di_pyield
    try:
        curva = curva_di_pyield(DATA)
        print(f"Curva do DI1 de {DATA}: {len(curva)} vértices, de {curva.index[0]} DU ({curva.iloc[0]:.2%}) "
              f"a {curva.index[-1]} DU ({curva.iloc[-1]:.2%})")
    except Exception as e:
        print(f"• Sem curva do DI1 via pyield ({e})")

res = processar_dia(pasta_dia, pasta_ant if pasta_ant and pasta_ant.exists() else None, data=DATA, params=params,
                    corretoras=corretoras, overrides=OVERRIDES, adtv_externo=adtv, curva_di=curva)
for a in res.avisos:
    print("•", a)
if Path(PASTA_HIST, "resumo.csv").exists():      # z de cada ativo contra o próprio histórico (reordena o Resumo)
    from fluxo_opcoes.validacao import z_do_dia
    res.resumo = z_do_dia(res.resumo, carregar_historico(PASTA_HIST)["resumo"], DATA)

u = res.universo.series
print(f"\n{len(u)} séries em {u['subjacente'].nunique()} ativos-objeto")
u.groupby("subjacente").agg(spot=("S", "first"), series=("tipo", "size"), iv_mediana=("iv", "median"))

## 3. Resumo por ativo

Score = maior entre |fluxo institucional|, |ΔOI dos titulares exceto travas| e o maior evento, dividido pelo ADTV.

In [ ]:
mostrar(res.resumo, ["ativo", "z_max", "score", "fluxo_demais", "n_estruturas", "n_direcionais", "n_financiamento",
                     "maior_estrutura", "n_blocos", "maior_bloco", "n_diretos", "dn_titulares_oi", "leitura_oi",
                     "n_concentradas", "pin"])

## 4. Corretoras

"Alto giro" = participação de pelo menos 10% nos negócios do ativo, compras e vendas equilibradas em contagem e muitas séries: formador ou varejo de massa. O mapeamento em `CORRETORAS` tem precedência. O fluxo exclui negócios diretos e pernas de box/conversão.

In [ ]:
mostrar(res.perfil.groupby("subjacente").head(8),
        ["subjacente", "corretora", "nome", "classe", "n_negocios", "n_series", "participacao", "desequilibrio"])

In [ ]:
fl = res.fluxo.assign(ordem=res.fluxo["dn_liquido"].abs()).sort_values(["subjacente", "ordem"], ascending=[True, False])
mostrar(fl.groupby("subjacente").head(5),
        ["subjacente", "corretora", "nome", "classe", "dn_liquido", "premio_liquido", "n_negocios"])

## 5. Estruturas, blocos e negócios diretos

A leitura é do ponto de vista da corretora indicada; "Espelho" traz quem fez o oposto quando também não é alto giro. A coluna `obs_oi` mostra o ΔOI de cada série envolvida no dia.

In [ ]:
COLS_EV = ["subjacente", "hora", "corretora", "nome_corretora", "contrapartes_txt", "espelho", "estrutura", "escopo", "leitura",
           "n_execucoes", "taxa_implicita", "dn_liquido", "dn_bruto", "pct_adtv", "abriu_posicao", "pernas_txt", "obs_oi"]
mostrar(res.estruturas, COLS_EV)

In [ ]:
mostrar(res.blocos, COLS_EV)

In [ ]:
d = res.diretos
if len(d) and "abriu_posicao" in d:
    d = d.sort_values(["abriu_posicao", "dn_bruto"], ascending=[False, False])
display(mostrar(d, ["subjacente", "hora", "corretora", "nome_corretora", "n_pernas", "n_execucoes", "estrutura", "leitura",
                    "dn_bruto", "pct_adtv", "confirmacao_oi", "abriu_posicao", "pernas_txt", "obs_oi"]))
fin = res.eventos[res.eventos["financiamento"].eq(True)] if len(res.eventos) else res.eventos
print("Financiamento via opções (box, conversão, sintético):")
mostrar(fin.sort_values("valor_final", ascending=False) if len(fin) else fin,
        ["subjacente", "tipo_evento", "corretora", "estrutura", "du_fin", "valor_final", "taxa_implicita", "taxa_di_ref",
         "spread_di", "pct_di", "n_execucoes", "pernas_txt"])

## 6. Posições em aberto

Titulares recebem o delta do ΔOI (lançadores têm o oposto). "Exceto travas" retira a parte do ΔOI explicada por posição travada (spreads e box).

In [ ]:
mostrar(res.pos_ativos.assign(o=res.pos_ativos["pct_adtv"]).sort_values("o", ascending=False),
        ["subjacente", "dn_titulares", "dn_titulares_ex_trava", "pct_adtv", "dn_calls", "dn_puts", "lanc_desc_calls",
         "lanc_desc_puts", "lanc_cob_calls", "nocional_trava", "n_concentradas", "leitura_oi"])

In [ ]:
mostrar(res.pos_series[res.pos_series["concentrada"]].sort_values("pct_adtv", ascending=False),
        ["subjacente", "ticker", "tipo", "strike", "vencimento", "oi", "d_oi", "d_oi_ef", "d_coberta", "d_travada",
         "d_descoberta", "d_n_clientes_1", "d_n_clientes_2", "motor", "dn_d_oi_ex_trava", "pct_adtv"])

## 7. Mapa do vencimento mais próximo

Strikes a até 5% do spot. Hedge por 1% supõe que os lançadores a seco façam delta hedge, o que nem sempre acontece.

In [ ]:
mapa = res.mapa
if len(mapa):
    display(mostrar(mapa[mapa["dist"].abs() <= 0.05],
                    ["subjacente", "vencimento", "du", "S", "strike", "dist", "oi_calls", "oi_puts", "desc_calls",
                     "desc_puts", "gama_1pct", "hedge_1pct_desc", "pin_candidato"]))
else:
    print("Nenhum vencimento dentro da janela do mapa.")

## 8. Painel em Excel

In [ ]:
arq = exportar_excel(res, f"painel_{DATA}.xlsx", aviso="DADOS SINTÉTICOS" if MODO_SINTETICO else None,
                     texto_br=PAINEL_TEXTO_BR)
print("Painel salvo em", arq)

## 9. Histórico e validação

Salve cada pregão processado: a partir de 3 pregões anteriores, a seção 2 já normaliza pelo ADTV médio (até 20). Com 60 ou mais, compare os sinais com o retorno anormal posterior (event study, com correção de Benjamini–Hochberg). Poucos sinais dão pouco poder estatístico: um efeito real pode não aparecer como significativo.

In [ ]:
from fluxo_opcoes.validacao import salvar_historico, adicionar_zscores, event_study

salvar_historico(res, PASTA_HIST)
hist = carregar_historico(PASTA_HIST)
n_pregoes = hist["resumo"]["data"].nunique()
print(n_pregoes, "pregão(ões) no histórico")

In [ ]:
if n_pregoes >= 60:
    z = adicionar_zscores(hist["resumo"], colunas=("fluxo_demais", "dn_titulares_oi"), janela=60, min_obs=20)
    sinais = []
    for col, nome in (("fluxo_demais", "fluxo institucional"), ("dn_titulares_oi", "ΔOI titulares")):
        s = z[z[f"z_{col}"].abs() >= 2].copy()
        s["direcao"] = np.sign(s[f"z_{col}"]).astype(int)
        s["tipo"] = f"{nome} |z|≥2"
        sinais.append(s[["data", "ativo", "direcao", "tipo"]])
    ev = hist["eventos"]
    ev = ev[ev["tipo_evento"].isin(["estrutura", "bloco"]) & (ev["direcao"] != 0)]
    sinais.append(ev.rename(columns={"subjacente": "ativo"}).assign(tipo="estruturas/blocos direcionais")
                  [["data", "ativo", "direcao", "tipo"]])
    display(event_study(pd.concat(sinais, ignore_index=True), hist["precos"], horizontes=(1, 5, 20),
                        benchmark="BOVA11"))
else:
    print("Event study disponível a partir de 60 pregões acumulados.")

## 10. Visão do período

Junta os pregões salvos no histórico: quantos dias cada ativo teve sinal, fluxo e ΔOI somados, eventos direcionais (sem financiamento) e séries com entrada concentrada em mais de um pregão. Sinal que se repete pesa mais do que um dia isolado.

In [ ]:
from fluxo_opcoes.validacao import resumo_periodo
from fluxo_opcoes.painel import exportar_periodo

RECALCULAR_SPREAD = False   # True uma vez: refaz o spread dos pregões já salvos com o DI1 de cada dia (pyield)
if RECALCULAR_SPREAD:
    from fluxo_opcoes.validacao import recalcular_spread_historico
    display(recalcular_spread_historico(PASTA_HIST))

per = resumo_periodo(PASTA_HIST)            # ou resumo_periodo(PASTA_HIST, inicio="2026-10-26")
display(mostrar(per["ativos"], ["ativo", "dias", "dias_anormais", "z_ult_oi", "seq_oi", "pct_oi_pos", "fluxo_pct_adtv",
                                 "oi_pct_adtv", "estruturas", "blocos", "diretos_abertura", "concentradas"]).head(20))
display(per["corretoras"].head(15) if len(per["corretoras"]) else "sem corretoras recorrentes")
print("Financiamento via opções por pregão (taxa implícita contra o DI do mesmo prazo):")
display(per["financiamento"] if len(per["financiamento"]) else "sem financiamento com taxa no histórico")
arq = exportar_periodo(per, f"painel_periodo_{DATA}.xlsx", texto_br=PAINEL_TEXTO_BR)
print("Visão do período salva em", arq)